In [1]:
%pip install spacy en_core_web_trf

ERROR: Could not find a version that satisfies the requirement en_core_web_trf (from versions: none)
ERROR: No matching distribution found for en_core_web_trf
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")

text = "In January 2024, VentureAlpha Partners invested $15 million into GreenFusion Technologies to support the expansion of their renewable energy projects."
doc = nlp(text)

# Extract entities
entities = {ent.label_: ent.text for ent in doc.ents}

# Use dependency parsing to find subject and object
investor = ""
target = ""
amount = entities.get("MONEY")
date = entities.get("DATE")

for token in doc:
    if token.lemma_ == "invest":
        for child in token.children:
            print(child, child.dep_)
        

# for token in doc:
#     if token.lemma_ == "invest":
#         # Subject (who invested)
#         for child in token.children:
#             if child.dep_ == "nsubj":
#                 investor = child.text
#         # Object (who received investment)
#         for prep in token.children:
#             if prep.dep_ == "prep" and prep.text == "into":
#                 for pobj in prep.children:
#                     if pobj.ent_type_ == "ORG":
#                         target = pobj.text

# result = {
#     "investor": investor,
#     "amount": amount,
#     "target": target,
#     "date": date
# }

# print(result)


In prep
, punct
Partners nsubj
million dobj
into prep
support advcl
. punct


In [12]:
from langchain.document_loaders import PyMuPDFLoader
import pymupdf4llm
from langchain.text_splitter import SentenceTransformersTokenTextSplitter, RecursiveCharacterTextSplitter, CharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import SentenceTransformerEmbeddings, HuggingFaceEmbeddings

class SimpleRAGPDF:
    def __init__(self, pdf_name: str):
        loader = PyMuPDFLoader(pdf_name)
        self.loaded_docs = loader.load()
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    def convert_to_markdown(self):
        self.mk_docs = pymupdf4llm.to_markdown(self.loaded_docs)
    
    def split_docs(self, type: str):
        splitter = RecursiveCharacterTextSplitter()
        self.split_docs = splitter.split_documents(self.loaded_docs)
        print(self.split_docs)
    
    def embed_docs(self):
        self.vc = Chroma.from_documents(self.split_docs, embedding=self.embeddings)
        self.retriever = self.vc.as_retriever()
    
    def get_relevant_docs(self, query: str):
        docs = self.vc.similarity_search(query, k=2)
        txt = [doc.page_content for doc in docs]
        return txt

rag = SimpleRAGPDF(pdf_name='test.pdf')
rag.split_docs(type='')
rag.embed_docs()

txt = rag.get_relevant_docs('Scope of Agreement')

del rag

print(txt)


[Document(metadata={'producer': 'Aspose.PDF for .NET 18.4', 'creator': 'Aspose Ltd.', 'creationdate': '', 'source': 'test.pdf', 'file_path': 'test.pdf', 'total_pages': 54, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-07-08T22:01:52-07:00', 'trapped': '', 'modDate': "D:20200708220152-07'00'", 'creationDate': '', 'page': 0}, page_content='Exhibit 10.1\n[***] = Certain confidential information contained in this document,\nmarked by brackets, has been omitted because it is both\n(i) not material and (ii) would likely be competitively harmful if publicly disclosed.\nMiltenyi Biotec-Bellicum\nSupply Agreement\n(Execution Copy March 27, 2019)\nSUPPLY AGREEMENT\n(MB Global Contract Number MBGCR 19001)\nThis Supply Agreement (this “Agreement”) is made and entered into, effective as of March 27, 2019 (the “Effective Date”), by and\nbetween Miltenyi Biotec GmbH, a German corporation having an address at Friedrich-Ebert-Str. 68, 51429 Bergisch Gla